# Day 64：毕业设计开发 Day 6：Docker 容器化部署上线、健康检查巡检、演示脚本与交付文档（Jupyter Notebook 实战手册）

> **所属阶段**：第 6 阶段 · 第六阶段：毕业设计与就业冲刺 ｜ **第 9 周**：星辰智脑毕业设计全栈开发与上线  
> **GitLab 工单**：`STARMIND-064` ｜ **开发分支**：`feat/day64-capstone-release-ops`  
> **配套 HTML5 课件**：`courseware/day64/index.html` ｜ **独立版本快照**：`versions/day64/`

---

## 🎙️ 老陈开场旁白

早啊各位！毕业设计开发最后一天（Day 64），咱们要把星辰智脑真正做成'拎包入住、一键上线、出故障能 3 分钟回滚'的企业级交付件！记住大厂交付标准：代码写完只算完成 70%，剩下 30% 是多阶段非 root Dockerfile、带健康依赖的 Docker Compose、SSE 优化版 Nginx、自动化上线巡检脚本，以及一份新同事照着敲就能跑起来的交付 Runbook 文档。今天咱们把这四件套全部工程化落地！

### 承上启下
- **昨日基线**：昨天我们跑出了 Ragas 四维质量报告、并发压测 SLA 和 Token 成本优化方案，今天是毕业设计开发收官日——把星辰智脑正式交付上线。
- **明日预告**：明天（Day 65）正式迎来毕业答辩日：15 分钟高分答辩结构、企业评委追问清单与防翻车演示验收脚本。


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day64'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-064）

星辰智脑完成全功能开发与压测后，需打包为标准容器镜像与编排清单部署至阿里云 ECS，同时提供上线前健康巡检脚本与运维交接文档（Runbook）。

**今日迭代目标：**
- 实现星辰智脑发布包生成器与合规校验器（Dockerfile / docker-compose.yml / nginx.conf / Delivery Runbook）。
- 实现上线前并发健康检查巡检器与回滚预案检查器，输出红绿灯上线放行报告。

**验收标准：**
- 生成的部署清单通过全部安全与流式配置审计项（非 root、HEALTHCHECK、proxy_buffering off、日志轮转）。
- 巡检器能并发检测所有服务探针，全绿输出 `GO_LIVE`，存在异常输出 `HOLD` 并定位故障节点。
- 自动生成包含启动命令、探针地址与一键回滚指令的企业级交付文档。

---

## 📚 课堂核心技术笔记

### 1. 生产 Docker Compose 三大铁律

1) 镜像严禁使用 `:latest`，必须锁定不可变语义化版本号（如 `starmind-api:1.0.0`），否则回滚时不知道回滚到哪一版；2) 所有容器必须配置 `logging.options.max-size` 防止日志撑爆磁盘；3) 数据库与上传文件必须挂载命名卷（Named Volume），容器重建数据不丢。

> **⚠️ 老陈避坑**：容器内修改配置文件重启后即丢失，所有密钥与开关必须通过 `.env` 或密钥管理服务注入。

### 2. 交付 Runbook 必须包含哪 4 块内容？

架构与端口清单、一键部署与环境变量说明、健康检查与常见故障排查表、精确到命令行的版本回滚预案。

> **⚠️ 老陈避坑**：不要只写'如有问题请回滚'，要把完整可复制粘贴的回滚命令写清楚。


## 💻 实操步骤 1：步骤一：实现星辰智脑容器化部署清单生成器与合规审计器

- **目标文件**：`versions/day64/src/day64_release_bundle.py`
- **老陈代码旁白**：一键渲染多阶段 Dockerfile、Docker Compose、SSE 优化 Nginx 配置与企业交付 Runbook，并执行自动化合规检查。


In [ ]:
"""
Day 64 - 模块一：星辰智脑容器化部署清单生成器、合规审计器与 Runbook 交付文档
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List


@dataclass
class ReleaseBundle:
    version: str
    dockerfile: str
    compose_yml: str
    nginx_conf: str
    runbook_md: str


def generate_release_bundle(version: str = "1.0.0") -> ReleaseBundle:
    dockerfile = (
        "FROM python:3.11-slim AS builder\n"
        "WORKDIR /build\n"
        "RUN python -m venv /opt/venv\n"
        "ENV PATH=\"/opt/venv/bin:$PATH\"\n"
        "COPY requirements.txt .\n"
        "RUN pip install --no-cache-dir -r requirements.txt\n\n"
        "FROM python:3.11-slim AS runtime\n"
        "RUN useradd -m -u 1000 starmind\n"
        "WORKDIR /app\n"
        "COPY --from=builder /opt/venv /opt/venv\n"
        "COPY src/ ./src/\n"
        "ENV PATH=\"/opt/venv/bin:$PATH\"\n"
        "USER starmind\n"
        "EXPOSE 8000\n"
        "HEALTHCHECK --interval=30s --timeout=5s --retries=3 CMD python -c \"import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/healthz')\"\n"
        "CMD [\"uvicorn\", \"src.day62_api_server:app\", \"--host\", \"0.0.0.0\", \"--port\", \"8000\"]\n"
    )

    compose_yml = (
        "services:\n"
        "  starmind-api:\n"
        f"    image: starmind-api:{version}\n"
        "    restart: unless-stopped\n"
        "    env_file: .env\n"
        "    ports:\n"
        "      - \"127.0.0.1:8000:8000\"\n"
        "    logging:\n"
        "      driver: json-file\n"
        "      options:\n"
        "        max-size: \"50m\"\n"
        "        max-file: \"5\"\n"
        "  nginx:\n"
        "    image: nginx:1.25-alpine\n"
        "    depends_on:\n"
        "      starmind-api:\n"
        "        condition: service_healthy\n"
        "    ports:\n"
        "      - \"443:443\"\n"
    )

    nginx_conf = (
        "location /api/v1/chat/stream {\n"
        "    proxy_pass http://starmind-api:8000;\n"
        "    proxy_http_version 1.1;\n"
        "    proxy_set_header Connection '';\n"
        "    proxy_buffering off;\n"
        "    proxy_cache off;\n"
        "    proxy_read_timeout 300s;\n"
        "}\n"
    )

    runbook_md = (
        f"# 星辰智脑 (StarMind) v{version} 生产交付运维手册 (Runbook)\n\n"
        "## 1. 一键部署命令\n"
        "```bash\n"
        f"docker compose up -d --build\n"
        "```\n\n"
        "## 2. 健康检查探针\n"
        "- 存活探针: `GET /healthz`\n"
        "- 就绪探针: `GET /readyz`\n\n"
        "## 3. 应急回滚命令\n"
        "```bash\n"
        "STARMIND_VERSION=0.9.9 docker compose up -d --no-build starmind-api\n"
        "```\n"
    )
    return ReleaseBundle(version, dockerfile, compose_yml, nginx_conf, runbook_md)


def audit_release_bundle(bundle: ReleaseBundle) -> Dict[str, List[str]]:
    """对部署交付物执行自动化安全与流式合规审计"""
    issues: List[str] = []
    if "AS builder" not in bundle.dockerfile or "USER starmind" not in bundle.dockerfile:
        issues.append("Dockerfile 必须采用多阶段构建并切换非 root 用户")
    if "HEALTHCHECK" not in bundle.dockerfile:
        issues.append("Dockerfile 缺少 HEALTHCHECK 指令")
    if ":latest" in bundle.compose_yml:
        issues.append("Docker Compose 严禁使用 :latest 镜像标签")
    if "max-size" not in bundle.compose_yml:
        issues.append("Docker Compose 缺少日志大小轮转限制")
    if "proxy_buffering off;" not in bundle.nginx_conf or "proxy_read_timeout 300s;" not in bundle.nginx_conf:
        issues.append("Nginx SSE 路由必须关闭 proxy_buffering 并配置长超时")
    if "应急回滚" not in bundle.runbook_md:
        issues.append("Runbook 交付文档缺少应急回滚预案")
    return {"passed": len(issues) == 0, "issues": issues}


if __name__ == "__main__":
    bundle = generate_release_bundle("1.0.0")
    print("部署合规审计:", audit_release_bundle(bundle))
    print("Runbook 预览:\n", bundle.runbook_md)


### 🔍 步骤 1 代码深度解析

通过代码化生成 + 静态合规审计，确保每次发布都不会遗漏非 root、镜像定版、日志轮转与 SSE 无缓冲等生产红线。

**预期运行输出参考：**
```text
部署合规审计: {'passed': True, 'issues': []}
Runbook 预览:
 # 星辰智脑 (StarMind) v1.0.0 生产交付运维手册 (Runbook)...
```


## 💻 实操步骤 2：步骤二：实现上线前并发健康检查巡检器与放行门禁

- **目标文件**：`versions/day64/src/day64_ops_preflight.py`
- **老陈代码旁白**：并发探测 API 存活、就绪、缓存与推理网关四大节点，输出上线放行判定与红绿灯巡检报告。


In [ ]:
"""
Day 64 - 模块二：生产上线前异步健康巡检引擎（Preflight Check）
"""
from __future__ import annotations

import asyncio
import time
from dataclasses import dataclass
from typing import Awaitable, Callable, Dict, List


@dataclass
class ProbeResult:
    service: str
    healthy: bool
    latency_ms: float
    detail: str


@dataclass
class PreflightReport:
    decision: str  # GO_LIVE | HOLD
    results: List[ProbeResult]
    failed_services: List[str]


async def run_preflight_checks(
    probes: Dict[str, Callable[[], Awaitable[bool]]],
    max_latency_ms: float = 500.0,
) -> PreflightReport:
    async def _check(name: str, fn: Callable[[], Awaitable[bool]]) -> ProbeResult:
        t0 = time.perf_counter()
        try:
            ok = await fn()
            dt = round((time.perf_counter() - t0) * 1000.0, 2)
            if not ok:
                return ProbeResult(name, False, dt, "探针返回非健康状态")
            if dt > max_latency_ms:
                return ProbeResult(name, False, dt, f"响应超时({dt}ms > {max_latency_ms}ms)")
            return ProbeResult(name, True, dt, "OK")
        except Exception as exc:
            dt = round((time.perf_counter() - t0) * 1000.0, 2)
            return ProbeResult(name, False, dt, f"探测异常: {exc}")

    results = await asyncio.gather(*(_check(k, v) for k, v in probes.items()))
    failed = [r.service for r in results if not r.healthy]
    decision = "GO_LIVE" if not failed else "HOLD"
    return PreflightReport(decision=decision, results=list(results), failed_services=failed)


if __name__ == "__main__":
    async def ok_probe() -> bool:
        await asyncio.sleep(0.005)
        return True

    rep = asyncio.run(
        run_preflight_checks(
            {
                "api_healthz": ok_probe,
                "api_readyz": ok_probe,
                "redis_cache": ok_probe,
                "llm_gateway": ok_probe,
            }
        )
    )
    print("上线巡检结论:", rep.decision, [(r.service, r.healthy, r.latency_ms) for r in rep.results])


### 🔍 步骤 2 代码深度解析

并发巡检把四项依赖探测压缩到一次网络往返耗时，任一服务宕机或响应超标立即返回 `HOLD` 阻止带病上线。

**预期运行输出参考：**
```text
上线巡检结论: GO_LIVE [('api_healthz', True, ...), ('api_readyz', True, ...), ('redis_cache', True, ...), ('llm_gateway', True, ...)]
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day64/tests/test_day64.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 64 单元测试：容器化部署合规审计与异步健康巡检门禁"""
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.day64_release_bundle import ReleaseBundle, audit_release_bundle, generate_release_bundle
from src.day64_ops_preflight import run_preflight_checks


def test_release_bundle_passes_audit():
    bundle = generate_release_bundle("1.0.0")
    res = audit_release_bundle(bundle)
    assert res["passed"] is True
    assert res["issues"] == []


def test_release_bundle_catches_insecure_configs():
    bad = ReleaseBundle(
        version="latest",
        dockerfile="FROM python:3.11\nUSER root\n",
        compose_yml="image: starmind-api:latest\n",
        nginx_conf="location / { proxy_pass http://app; }\n",
        runbook_md="# 简陋文档\n",
    )
    res = audit_release_bundle(bad)
    assert res["passed"] is False
    assert len(res["issues"]) >= 5


def test_preflight_go_live_and_hold():
    async def ok():
        return True

    async def down():
        return False

    r_ok = asyncio.run(run_preflight_checks({"api": ok, "db": ok}))
    assert r_ok.decision == "GO_LIVE" and r_ok.failed_services == []

    r_bad = asyncio.run(run_preflight_checks({"api": ok, "llm": down}))
    assert r_bad.decision == "HOLD" and r_bad.failed_services == ["llm"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：编写 .env 必填生产环境变量校验器

**题目要求：**
实现 `check_required_env(env_map, required_keys)`，返回缺失或为空值的环境变量列表。

**思路提示：** 列表推导 `[k for k in required_keys if not str(env_map.get(k, '')).strip()]`。


In [ ]:
def check_required_env(env_map: dict, required_keys: list[str]) -> list[str]:
    return [k for k in required_keys if not str(env_map.get(k, "")).strip()]


missing = check_required_env({"LLM_API_KEY": "sk-xxx", "JWT_SECRET": ""}, ["LLM_API_KEY", "JWT_SECRET", "DB_URL"])
assert missing == ["JWT_SECRET", "DB_URL"]
print("生产环境变量缺失检测通过:", missing)


**答案解析与评分标准：**

上线前强校验 `.env` 必填项能避免容器启动 10 分钟后才因空密钥报错。


### 作业 2（企业实战）：企业实战：生成带 SHA-256 校验和的发布产物清单

**题目要求：**
编写 `build_checksum_manifest(files: dict[str, str]) -> dict[str, str]`，为每个交付文件内容计算 SHA-256 摘要。

**思路提示：** 使用标准库 `hashlib.sha256(content.encode('utf-8')).hexdigest()`。


In [ ]:
import hashlib


def build_checksum_manifest(files: dict[str, str]) -> dict[str, str]:
    return {name: hashlib.sha256(body.encode("utf-8")).hexdigest() for name, body in files.items()}


m = build_checksum_manifest({"Dockerfile": "FROM python:3.11-slim"})
assert len(m["Dockerfile"]) == 64
print("发布产物 SHA-256 清单生成通过:", m)


**答案解析与评分标准：**

SHA-256 摘要可确保测试环境验收通过的配置文件与生产环境部署的完全一致，防篡改。


### 作业 3（架构进阶）：架构进阶：设计灰度发布流量比例校验器（10% -> 50% -> 100%）

**题目要求：**
编写 `next_canary_stage(current_pct, error_rate)`：若错误率 `> 0.01` 返回 `0`（立即回滚），否则按 `0->10->50->100` 阶梯放量。

**思路提示：** 先判断错误率超标直接回滚至 0，再用字典或条件映射下一档比例。


In [ ]:
def next_canary_stage(current_pct: int, error_rate: float) -> int:
    if error_rate > 0.01:
        return 0
    steps = {0: 10, 10: 50, 50: 100, 100: 100}
    return steps.get(current_pct, 0)


assert next_canary_stage(10, 0.002) == 50
assert next_canary_stage(50, 0.03) == 0
print("灰度放量与自动回滚状态机通过")


**答案解析与评分标准：**

灰度阶梯放量配合错误率自动回滚，是高可用 AI 网关发布新模型或新 Prompt 的标准动作。


---

## 🎯 老陈收尾复盘

漂亮！星辰智脑 6 天毕业设计开发全部收官——从底座到 RAG、多 Agent+HITL、FastAPI+SSE、Ragas+压测+降本，再到今天的容器化交付与健康巡检，整套工程已经达到国内一线大厂准生产标准！明天咱们站上答辩台！
